In [1]:
import os
import torch
from torch.utils.data import DataLoader
from torch.optim import AdamW
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from tqdm import tqdm
from sklearn.metrics import classification_report


/home/andresmtr/miniconda3/envs/PruebaBRM/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Activar CUDA Launch Blocking para depuración
os.environ['CUDA_LAUNCH_BLOCKING'] = '1'

In [3]:
# Verificacion del uso de cuda
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


# Carge de datos

In [4]:
dataset = load_dataset("alexcom/analisis-sentimientos-textos-turisitcos-mx-polaridad")

In [5]:
dataset = dataset["train"].train_test_split(test_size=0.2, seed=42)

# Preparar los datos para la tokenizacion

In [6]:
tokenizer = AutoTokenizer.from_pretrained("PlanTL-GOB-ES/roberta-base-bne")

In [7]:
def preprocess(batch):
    batch["label"] = [label - 1 for label in batch["label"]]  # Resta 1 a cada etiqueta
    return tokenizer(batch["text"], truncation=True, padding="max_length", max_length=128)



In [8]:
encoded_dataset = dataset.map(preprocess, batched=True)
encoded_dataset = encoded_dataset.rename_column("label", "labels")
encoded_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

Map: 100%|██████████| 35239/35239 [00:02<00:00, 15820.68 examples/s]


# Crear los data Loaders

In [9]:
train_dataloader = DataLoader(encoded_dataset["train"], batch_size=16, shuffle=True)
eval_dataloader = DataLoader(encoded_dataset["test"], batch_size=16)

# Cargar modelo preeentrenado

In [10]:
model = AutoModelForSequenceClassification.from_pretrained(
    "PlanTL-GOB-ES/roberta-base-bne",
    num_labels=5  # Etiquetas de 1 a 5
)

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at PlanTL-GOB-ES/roberta-base-bne and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


# Configurar optmizador y la perdida

In [11]:
optimizer = AdamW(model.parameters(), lr=2e-5)

# Entrenamiento

In [ ]:
model.to(device)

model.train()
for epoch in range(3):
    loop = tqdm(train_dataloader, leave=True)
    for batch in loop:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        optimizer.zero_grad()
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        loss.backward()
        optimizer.step()

        loop.set_description(f"Epoch {epoch}")
        loop.set_postfix(loss=loss.item())


Epoch 0:   6%|▋         | 563/8810 [00:51<12:32, 10.96it/s, loss=0.823]

# Evaluacion

In [ ]:

model.eval()
predictions, references = [], []

with torch.no_grad():
    for batch in eval_dataloader:
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**batch)
        logits = outputs.logits
        preds = torch.argmax(logits, axis=1)
        predictions.extend(preds.cpu().numpy())
        references.extend(batch["labels"].cpu().numpy())

print(classification_report(references, predictions))